# Sync Author Curations

Syncs author curations (add/remove works, set display_name/full_name) from the users Heroku Postgres database to local Databricks Delta tables.

**Sources** (typed views over `openalex_users.public.curations`):
- `work_authorship_add_curations` — add a work to an author profile at a specific slot (`property = 'authorships[<N>].author.id'`); exposes `author_sequence INT`
- `work_authorship_remove_curations` — non-positional sticky disclaim (`property = 'authorships.author.id'`)
- `author_curations_set_display_name` — set a display_name for an author profile
- `author_curations_set_full_name` — set a full_name (used in matching) for an author profile

**Targets** (split by action / consumer):
- `openalex.works.work_author_add_curations` — flat per-curation rows; `raw_author_name` captured at sync time from `work_authors @ (work_id, author_sequence)` for re-ingest resilience
- `openalex.works.work_author_remove_curations` — flat per-curation rows; non-positional
- `openalex.authors.author_names_curations` — per-author display_name / full_name (applied during profile assembly / matching)

Modeled on `SyncRasCurations.ipynb` with a guardrail cell added before each MERGE so a silent empty/short source can't mass-delete the Delta table. The `WHEN NOT MATCHED BY SOURCE THEN DELETE` clause is what propagates user-initiated retractions (`DELETE /curations/{id}`), so it stays — the guardrail just refuses to fire it on a suspicious source.

**Dep ordering**: this notebook runs after `Author_Affiliations` (which populates `work_authors`), so the name-capture join on the add side has data to read.

**v0 application status**: works curations have apply logic in `ApplyWorkAuthorCurations`. Names curations sync to Delta but are not yet applied (`set` action is currently rejected by `POST /curations` validation, so the names views and target will stay empty until producer-side support lands).

## Create target tables (idempotent)

In [ ]:
%sql
-- Add-side curations: positional. raw_author_name is captured from work_authors at
-- first INSERT and frozen thereafter (audit/diagnostic only — not used as an apply
-- guard). applied_at gates apply-once semantics: NULL until the apply notebook has
-- merged the row, then stamped so subsequent runs skip it.
CREATE TABLE IF NOT EXISTS openalex.works.work_author_add_curations (
    curation_id        STRING NOT NULL,
    user_id            STRING NOT NULL,
    author_id          BIGINT NOT NULL,
    work_id            BIGINT NOT NULL,
    author_sequence    INT    NOT NULL,
    raw_author_name    STRING,
    created            TIMESTAMP,
    updated_datetime   TIMESTAMP,
    applied_at         TIMESTAMP
)
USING DELTA
CLUSTER BY (work_id)

In [ ]:
%sql
-- Remove-side curations: non-positional sticky disclaim. MatchAuthors anti-joins on
-- (work_id, author_id) regardless of slot; the apply step NULLs the author_id at any
-- matching slot. applied_at gates apply-once semantics on the remove side too.
CREATE TABLE IF NOT EXISTS openalex.works.work_author_remove_curations (
    curation_id        STRING NOT NULL,
    user_id            STRING NOT NULL,
    author_id          BIGINT NOT NULL,
    work_id            BIGINT NOT NULL,
    created            TIMESTAMP,
    updated_datetime   TIMESTAMP,
    applied_at         TIMESTAMP
)
USING DELTA
CLUSTER BY (author_id)

In [ ]:
%sql
-- Per-author display_name / full_name curations.
-- Latest-wins by source `created` when multiple curators set the same property
-- (aggregation handled in the MERGE source query below).
CREATE TABLE IF NOT EXISTS openalex.authors.author_names_curations (
    author_id             BIGINT NOT NULL,
    curated_display_name  STRING,
    curated_full_name     STRING,
    updated_datetime      TIMESTAMP
)
USING DELTA
CLUSTER BY (author_id)

## Sync works curations

In [ ]:
%sql
-- Guardrail: refuse to MERGE if the source has unexpectedly few rows.
-- Counts cover BOTH the add and remove views (we want to fail-stop on either side
-- emptying out, since a single guardrail cell precedes both MERGEs).
-- The empty-source check is conditional on the target already holding rows,
-- so the legitimate startup state (both 0) doesn't fail; only "we had data
-- and now the source is empty" fails.
-- Set guardrails_override=true on the job to bypass the decline check
-- (the empty-when-target-nonempty check is unconditional).
DECLARE OR REPLACE VARIABLE new_count BIGINT;
DECLARE OR REPLACE VARIABLE current_count BIGINT;
DECLARE OR REPLACE VARIABLE allowed_decline BIGINT DEFAULT 10;

SET VARIABLE new_count = (
  SELECT COUNT(*) FROM (
    SELECT curation_id FROM openalex_users.public.work_authorship_add_curations
    UNION ALL
    SELECT curation_id FROM openalex_users.public.work_authorship_remove_curations
  )
);
SET VARIABLE current_count = (
  SELECT (SELECT COUNT(*) FROM openalex.works.work_author_add_curations)
       + (SELECT COUNT(*) FROM openalex.works.work_author_remove_curations)
);

SELECT
  new_count AS new_curations,
  current_count AS current_curations,
  ROUND(new_count * 100.0 / NULLIF(current_count, 0), 2) AS pct_of_current,
  COALESCE(:guardrails_override, 'false') AS guardrails_override;

SELECT CASE
  WHEN current_count > 0 AND new_count = 0
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source has 0 work-author curations but targets hold ',
    CAST(current_count AS STRING),
    ' rows. Aborting to prevent mass delete. Investigate the source views before re-running.'))
END;

SELECT CASE
  WHEN current_count > 0
   AND new_count < current_count - allowed_decline
   AND LOWER(COALESCE(:guardrails_override, 'false')) <> 'true'
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source curation count declined by ',
    CAST(current_count - new_count AS STRING),
    ' (', CAST(new_count AS STRING), ' vs ', CAST(current_count AS STRING),
    '), exceeding allowed decline of ', CAST(allowed_decline AS STRING),
    '. Set guardrails_override=true on the job to bypass.'))
END;

In [ ]:
%sql
-- Preview what will be synced: each curation row paired with the raw_author_name
-- captured from work_authors at the (work_id, author_sequence) the curator clicked
-- (NULL on the remove side since remove is non-positional).
SELECT
    'add'  AS action_kind,
    v.curation_id, v.user_id, v.author_id, v.work_id, v.author_sequence,
    wa.raw_author_name AS captured_raw_author_name,
    v.created
FROM openalex_users.public.work_authorship_add_curations v
LEFT JOIN openalex.works.work_authors wa
  ON wa.work_id = v.work_id
 AND wa.author_sequence = v.author_sequence
UNION ALL
SELECT
    'remove' AS action_kind,
    curation_id, user_id, author_id, work_id,
    CAST(NULL AS INT) AS author_sequence,
    CAST(NULL AS STRING) AS captured_raw_author_name,
    created
FROM openalex_users.public.work_authorship_remove_curations

In [ ]:
%sql
-- MERGE add-side curations. JOINs work_authors at sync time to capture
-- raw_author_name at the curator's clicked slot — frozen at first INSERT.
-- INSERT/UPDATE column lists explicit so applied_at is preserved (NULL on
-- first INSERT, untouched by subsequent UPDATEs).
MERGE INTO openalex.works.work_author_add_curations AS target
USING (
    SELECT
        v.curation_id,
        v.user_id,
        v.author_id,
        v.work_id,
        v.author_sequence,
        wa.raw_author_name,
        v.created,
        CURRENT_TIMESTAMP() AS updated_datetime
    FROM openalex_users.public.work_authorship_add_curations v
    LEFT JOIN openalex.works.work_authors wa
      ON wa.work_id = v.work_id
     AND wa.author_sequence = v.author_sequence
) AS source
ON target.curation_id = source.curation_id
WHEN MATCHED THEN UPDATE SET
    target.user_id          = source.user_id,
    target.author_id        = source.author_id,
    target.work_id          = source.work_id,
    target.author_sequence  = source.author_sequence,
    target.created          = source.created,
    target.updated_datetime = source.updated_datetime
    -- DELIBERATELY NOT raw_author_name — frozen at first INSERT
    -- DELIBERATELY NOT applied_at — written only by the apply notebook
WHEN NOT MATCHED THEN INSERT (
    curation_id, user_id, author_id, work_id, author_sequence,
    raw_author_name, created, updated_datetime
) VALUES (
    source.curation_id, source.user_id, source.author_id, source.work_id, source.author_sequence,
    source.raw_author_name, source.created, source.updated_datetime
)
WHEN NOT MATCHED BY SOURCE THEN DELETE

In [ ]:
%sql
-- MERGE remove-side curations. Non-positional, no work_authors join.
-- Explicit column lists preserve applied_at (NULL on first INSERT, untouched
-- by subsequent UPDATEs — the apply notebook owns that column).
MERGE INTO openalex.works.work_author_remove_curations AS target
USING (
    SELECT
        curation_id, user_id, author_id, work_id, created,
        CURRENT_TIMESTAMP() AS updated_datetime
    FROM openalex_users.public.work_authorship_remove_curations
) AS source
ON target.curation_id = source.curation_id
WHEN MATCHED THEN UPDATE SET
    target.user_id          = source.user_id,
    target.author_id        = source.author_id,
    target.work_id          = source.work_id,
    target.created          = source.created,
    target.updated_datetime = source.updated_datetime
    -- DELIBERATELY NOT applied_at — written only by the apply notebook
WHEN NOT MATCHED THEN INSERT (
    curation_id, user_id, author_id, work_id, created, updated_datetime
) VALUES (
    source.curation_id, source.user_id, source.author_id, source.work_id, source.created, source.updated_datetime
)
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Sync names curations

In [ ]:
%sql
-- Guardrail: same shape as the works guardrail. The source for names is the
-- UNION of distinct author_ids across the two `set_*name` views.
DECLARE OR REPLACE VARIABLE new_count BIGINT;
DECLARE OR REPLACE VARIABLE current_count BIGINT;
DECLARE OR REPLACE VARIABLE allowed_decline BIGINT DEFAULT 10;

SET VARIABLE new_count = (
  SELECT COUNT(DISTINCT author_id) FROM (
    SELECT author_id FROM openalex_users.public.author_curations_set_display_name
    UNION
    SELECT author_id FROM openalex_users.public.author_curations_set_full_name
  )
);
SET VARIABLE current_count = (SELECT COUNT(*) FROM openalex.authors.author_names_curations);

SELECT
  new_count AS new_authors,
  current_count AS current_authors,
  ROUND(new_count * 100.0 / NULLIF(current_count, 0), 2) AS pct_of_current,
  COALESCE(:guardrails_override, 'false') AS guardrails_override;

SELECT CASE
  WHEN current_count > 0 AND new_count = 0
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source has 0 author name-curations but target has ',
    CAST(current_count AS STRING),
    ' rows. Aborting to prevent mass delete. Investigate the source views before re-running.'))
END;

SELECT CASE
  WHEN current_count > 0
   AND new_count < current_count - allowed_decline
   AND LOWER(COALESCE(:guardrails_override, 'false')) <> 'true'
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source author count declined by ',
    CAST(current_count - new_count AS STRING),
    ' (', CAST(new_count AS STRING), ' vs ', CAST(current_count AS STRING),
    '), exceeding allowed decline of ', CAST(allowed_decline AS STRING),
    '. Set guardrails_override=true on the job to bypass.'))
END;

In [ ]:
%sql
-- Preview what will be synced (latest-wins per author for each name property)
WITH latest_display_name AS (
  SELECT author_id, new_display_name AS curated_display_name
  FROM (
    SELECT
      author_id,
      new_display_name,
      ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
    FROM openalex_users.public.author_curations_set_display_name
  )
  WHERE rn = 1
),
latest_full_name AS (
  SELECT author_id, new_full_name AS curated_full_name
  FROM (
    SELECT
      author_id,
      new_full_name,
      ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
    FROM openalex_users.public.author_curations_set_full_name
  )
  WHERE rn = 1
)
SELECT
  COALESCE(d.author_id, f.author_id) AS author_id,
  d.curated_display_name,
  f.curated_full_name
FROM latest_display_name d
FULL OUTER JOIN latest_full_name f USING (author_id)

In [ ]:
%sql
-- MERGE names curations (handles inserts, updates, AND deletes via NOT MATCHED BY SOURCE)
MERGE INTO openalex.authors.author_names_curations AS target
USING (
  WITH latest_display_name AS (
    SELECT author_id, new_display_name AS curated_display_name
    FROM (
      SELECT
        author_id,
        new_display_name,
        ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
      FROM openalex_users.public.author_curations_set_display_name
    )
    WHERE rn = 1
  ),
  latest_full_name AS (
    SELECT author_id, new_full_name AS curated_full_name
    FROM (
      SELECT
        author_id,
        new_full_name,
        ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
      FROM openalex_users.public.author_curations_set_full_name
    )
    WHERE rn = 1
  )
  SELECT
    COALESCE(d.author_id, f.author_id) AS author_id,
    d.curated_display_name,
    f.curated_full_name,
    CURRENT_TIMESTAMP() AS updated_datetime
  FROM latest_display_name d
  FULL OUTER JOIN latest_full_name f USING (author_id)
) AS source
ON target.author_id = source.author_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Verify sync results

In [ ]:
%sql
-- Works targets: row counts and name-capture coverage
SELECT
  (SELECT COUNT(*)                      FROM openalex.works.work_author_add_curations)    AS add_curation_rows,
  (SELECT COUNT(raw_author_name)        FROM openalex.works.work_author_add_curations)    AS add_rows_with_captured_name,
  (SELECT COUNT(*)                      FROM openalex.works.work_author_remove_curations) AS remove_curation_rows,
  (SELECT MAX(updated_datetime)         FROM openalex.works.work_author_add_curations)    AS add_last_sync,
  (SELECT MAX(updated_datetime)         FROM openalex.works.work_author_remove_curations) AS remove_last_sync

In [ ]:
%sql
-- Sample of recent curations from both sides (action_kind discriminator)
SELECT 'add' AS action_kind,
       curation_id, user_id, author_id, work_id, author_sequence, raw_author_name, created, updated_datetime
FROM openalex.works.work_author_add_curations
UNION ALL
SELECT 'remove' AS action_kind,
       curation_id, user_id, author_id, work_id,
       CAST(NULL AS INT) AS author_sequence,
       CAST(NULL AS STRING) AS raw_author_name,
       created, updated_datetime
FROM openalex.works.work_author_remove_curations
ORDER BY updated_datetime DESC
LIMIT 100

In [ ]:
%sql
-- Names target: row counts
SELECT
  COUNT(*)                                                AS total_curated_authors,
  COUNT(curated_display_name)                             AS total_display_names,
  COUNT(curated_full_name)                                AS total_full_names,
  MAX(updated_datetime)                                   AS last_sync
FROM openalex.authors.author_names_curations

In [ ]:
%sql
-- Sample of curated authors (names)
SELECT * FROM openalex.authors.author_names_curations
ORDER BY updated_datetime DESC
LIMIT 100